### Stations

In [8]:
import requests
import pandas as pd
import time

In [21]:

url = "https://hubeau.eaufrance.fr/api/v1/niveaux_nappes/stations"
params = {"size": 5000, "code_departement": 24}
headers = {"accept": "application/json"}

response = requests.get(url, params=params, headers=headers)
response.raise_for_status() 

data = response.json()
df_station = pd.DataFrame(data["data"])
df_station = df_station.rename(columns={"code_bss": "code_station", "x": "longitude", "y": "latitude"})

print(data.get("count"), "résultats au total")
display(df_station.describe())
display(df_station.head())

234 résultats au total


,longitude,latitude,nb_mesures_piezo,profondeur_investigation
count,234.000000,234.000000,234.000000,228.000000
mean,0.688151,45.042011,1734.965812,166.714035
std,0.321903,0.246209,2663.643967,155.380297
min,0.016523,44.588493,0.000000,0.000000
25%,0.419140,44.843795,1.000000,58.125000
50%,0.690197,45.006581,14.500000,111.000000
75%,0.926058,45.224775,3868.750000,245.750000
max,1.388338,45.672371,8301.000000,800.000000


,code_station,urn_bss,date_debut_mesure,date_fin_mesure,code_commune_insee,nom_commune,longitude,latitude,codes_bdlisa,urns_bdlisa,...,altitude_station,nb_mesures_piezo,code_departement,nom_departement,libelle_pe,profondeur_investigation,codes_masse_eau_edl,noms_masse_eau_edl,urns_masse_eau_edl,date_maj
0,08081X0026/SE.20,http://services.ades.eaufrance.fr/pointeau/080...,1996-09-05,2026-07-14,24255,Marquay,1.148945,44.923232,[348AA01],[http://reseau.eaufrance.fr/geotraitements/bdl...,...,154.0,7959,24,Dordogne,COMBE BOYER (MARQUAY - 24),87.0,[FG108],"[Calcaires, calcaires crayeux, grès, sables et...",[http://www.sandre.eaufrance.fr/geo/MasseDEauS...,Tue Mar 18 21:37:09 CET 2025
1,07598X0005/A81,http://services.ades.eaufrance.fr/pointeau/075...,1996-07-30,2026-07-13,24555,Tourtoirac,1.070420,45.249720,[358AE07],[http://reseau.eaufrance.fr/geotraitements/bdl...,...,165.8,7212,24,Dordogne,ROUGERIE A81 (TOURTOIRAC-24),66.0,[FG003],[Calcaires du Jurassique moyen des bassins ver...,[http://www.sandre.eaufrance.fr/geo/MasseDEauS...,Tue Sep 03 07:12:40 CEST 2024
2,07847X0010/F1,http://services.ades.eaufrance.fr/pointeau/078...,1986-10-01,1986-12-10,24085,La Cassagne,1.318102,45.045604,[358AE07],[http://reseau.eaufrance.fr/geotraitements/bdl...,...,156.87,3,24,Dordogne,F1,200.0,[FG040],[Calcaires du Jurassique moyen des Causses du ...,[http://www.sandre.eaufrance.fr/geo/MasseDEauS...,Fri Jun 28 07:31:38 CEST 2024
3,07345X0020/F1,http://services.ades.eaufrance.fr/pointeau/073...,2002-07-11,2012-04-04,24119,Cherval,0.403764,45.376158,None,[],...,148.0,566,24,Dordogne,LA FEUILLADE (CHERVAL-24),135.0,[FG073A],[Multicouches calcaire captif du Turonien-Coni...,[http://www.sandre.eaufrance.fr/geo/MasseDEauS...,Fri Jun 28 07:31:38 CEST 2024
4,07818X0030/F1,http://services.ades.eaufrance.fr/pointeau/078...,1996-08-30,1996-08-30,24462,Saint-Médard-de-Mussidan,0.332473,45.023305,None,[],...,57.0,1,24,Dordogne,NaN,70.0,None,None,[],Fri Jun 28 07:31:38 CEST 2024


### Piezometre

In [ ]:
url = "https://hubeau.eaufrance.fr/api/v1/niveaux_nappes/chroniques"
headers = {"accept": "application/json"}

def paginer(first_url: str,
               code_station: str) -> pd.DataFrame:
    data_page = []
    next_url = first_url
    params = {
        "code_bss": code_station,
        "page": 1,
        "size": 5000
    }

    while next_url:
        try:
            r = requests.get(next_url, params=params, headers=headers, timeout=(5, 30))
            r.raise_for_status()
            print (r.url)

        except requests.exceptions.Timeout:
            print("Timeout, nouvelle tentative...")
            time.sleep(2)
            continue 

        data = r.json()
        data_page.extend(data["data"])
        next_url = data.get("next")
        params = None

        time.sleep(1)
        
    df = pd.DataFrame(data_page)
    return df

df_p= []

for i, (code_station, lat, lon) in enumerate(zip(df_station["code_station"], df_station["latitude"], df_station["longitude"])):
    if i>0 :
        break

    print (f"----- Station: {code_station}, latitude:{lat}-longitude:{lon}")
    df_cache = paginer(url, code_station)
    df_p.append(df_cache)


piezometre_df = pd.concat(df_p, ignore_index=True)

print(len(piezometre_df), "résultats au total")

date_debut= piezometre_df['date_mesure'].min()
date_fin= piezometre_df['date_mesure'].max()

print(f"Début de la mesure {date_debut}, fin de la mesure {date_fin}")
print("\n")
display(piezometre_df)

----- Station: 08081X0026/SE.20, latitude:44.923232442-longitude:1.148944586
https://hubeau.eaufrance.fr/api/v1/niveaux_nappes/chroniques?code_bss=08081X0026%2FSE.20&page=1&size=5000
https://hubeau.eaufrance.fr/api/v1/niveaux_nappes/chroniques?code_bss=08081X0026/SE.20&page=2&size=5000
7960 résultats au total
Début de la mesure 1996-09-05, fin de la mesure 2026-07-15




,code_bss,bss_id,urn_bss,date_mesure,timestamp_mesure,niveau_nappe_eau,mode_obtention,statut,qualification,code_continuite,nom_continuite,code_producteur,nom_producteur,code_nature_mesure,nom_nature_mesure,profondeur_nappe
0,08081X0026/SE.20,BSS001YSAR,http://services.ades.eaufrance.fr/pointeau/080...,1996-09-05,841881600000,152.52,Valeur mesurée,Donnée contrôlée niveau 1,Correcte,2,Point lié au point précédent,327,Service Géologique Régional d'Aquitaine (327),0,Inconnue,1.40
1,08081X0026/SE.20,BSS001YSAR,http://services.ades.eaufrance.fr/pointeau/080...,1998-05-14,895104000000,152.93,Valeur mesurée,Donnée contrôlée niveau 1,Correcte,2,Point lié au point précédent,327,Service Géologique Régional d'Aquitaine (327),0,Inconnue,0.99
2,08081X0026/SE.20,BSS001YSAR,http://services.ades.eaufrance.fr/pointeau/080...,1999-08-17,934848000000,152.93,Valeur mesurée,Donnée contrôlée niveau 1,Correcte,2,Point lié au point précédent,327,Service Géologique Régional d'Aquitaine (327),0,Inconnue,0.99
3,08081X0026/SE.20,BSS001YSAR,http://services.ades.eaufrance.fr/pointeau/080...,2000-08-24,967075200000,152.98,Valeur mesurée,Donnée contrôlée niveau 1,Correcte,2,Point lié au point précédent,327,Service Géologique Régional d'Aquitaine (327),0,Inconnue,0.94
4,08081X0026/SE.20,BSS001YSAR,http://services.ades.eaufrance.fr/pointeau/080...,2001-10-17,1003276800000,152.78,Valeur mesurée,Donnée contrôlée niveau 1,Correcte,2,Point lié au point précédent,327,Service Géologique Régional d'Aquitaine (327),0,Inconnue,1.14
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7955,08081X0026/SE.20,BSS001YSAR,http://services.ades.eaufrance.fr/pointeau/080...,2026-07-11,1783728000000,152.69,Valeur mesurée,Donnée brute,Non qualifié,2,Point lié au point précédent,327,Service Géologique Régional d'Aquitaine (327),N,Naturel,1.23
7956,08081X0026/SE.20,BSS001YSAR,http://services.ades.eaufrance.fr/pointeau/080...,2026-07-12,1783814400000,152.64,Valeur mesurée,Donnée brute,Non qualifié,2,Point lié au point précédent,327,Service Géologique Régional d'Aquitaine (327),N,Naturel,1.28
7957,08081X0026/SE.20,BSS001YSAR,http://services.ades.eaufrance.fr/pointeau/080...,2026-07-13,1783900800000,152.64,Valeur mesurée,Donnée brute,Non qualifié,2,Point lié au point précédent,327,Service Géologique Régional d'Aquitaine (327),N,Naturel,1.28
7958,08081X0026/SE.20,BSS001YSAR,http://services.ades.eaufrance.fr/pointeau/080...,2026-07-14,1783990800000,152.64,Valeur mesurée,Donnée brute,Non qualifié,2,Point lié au point précédent,327,Service Géologique Régional d'Aquitaine (327),N,Naturel,1.28
